In [3]:
import sys
import pandas as pd 
import numpy as np 

sys.path.append("/var/www/python/Qingcheng/nighthawk/")
from nighthawk.util import bigquery_functions, sql_functions


In [4]:
# 1. Find out the worst, 5% worst LMP for DA and RT LMP for node 348 in last 1 year? What is worst, 5% Loss and Congestion component
#For node 348, find out the 5 min LMP for mar 17, 2021 between hour 5 and 
df_rt= sql_functions.download_df_from_sql_db(f'''select * from spp_lmp.settlement_location_rt_hourly where dt>='2025-03-24' and node_num=348 ''')
df_da= sql_functions.download_df_from_sql_db(f'''select * from spp_lmp.settlement_location_da_hourly where dt>='2025-03-24' and node_num=348 ''')
# --- Q1: Worst & 5% worst LMP ---
print("=== RT LMP ===")
print("Worst RT LMP:", df_rt['rt_value'].min())
print("5% worst RT LMP:", df_rt['rt_value'].quantile(0.05))

print("\n=== DA LMP ===")
print("Worst DA LMP:", df_da['da_value'].min())
print("5% worst DA LMP:", df_da['da_value'].quantile(0.05))

# --- Q2: Worst & 5% worst Congestion and Loss ---
for label, df in [("RT", df_rt), ("DA", df_da)]:
    print(f"\n=== {label} Congestion ===")
    print("Worst:", df['congestional_value'].min())
    print("5% worst:", df['congestional_value'].quantile(0.05))

    print(f"\n=== {label} Loss ===")
    print("Worst:", df['marginalloss_value'].min())
    print("5% worst:", df['marginalloss_value'].quantile(0.05))

df_5min = sql_functions.download_df_from_sql_db(
    "select * from spp_lmp.settlement_location_rt_5_min where dt = '2021-03-17' and node_num = 348 and hr in (5, 6)")
print("\n=== 5-min LMP Mar 17 2021, Hr 5-6 ===")
print(df_5min)


=== RT LMP ===
Worst RT LMP: -206.2668
5% worst RT LMP: -16.447490000000002

=== DA LMP ===
Worst DA LMP: -29.035
5% worst DA LMP: -7.9013

=== RT Congestion ===
Worst: -324.5938
5% worst: -38.645329999999994

=== RT Loss ===
Worst: -107.6644
5% worst: -5.7044500000000005

=== DA Congestion ===
Worst: -222.9091
5% worst: -29.58875

=== DA Loss ===
Worst: -27.3649
5% worst: -6.01

=== 5-min LMP Mar 17 2021, Hr 5-6 ===
Empty DataFrame
Columns: [forecast_time, dt, hr, settlement_location, node_num, rt_value, marginalloss_value, congestional_value]
Index: []


In [5]:
# 2. acutal wind and load

df_wind= sql_functions.download_df_from_sql_db('''select * from spp_physical.MTWF where ForecastTime='2018-03-20 00:00:00' and IntervalEnd like '2018-03-21%%' ''')
df_load= sql_functions.download_df_from_sql_db('''select * from spp_physical.MTLF where ForecastTime='2018-03-20 00:00:00' and IntervalEnd like '2018-03-21%%' ''')
actual_wind= sql_functions.download_df_from_sql_db('''select * from spp_physical.spp_latest_wind_forecast where dt='2018-03-21' ''')
actual_load= sql_functions.download_df_from_sql_db('''select * from spp_physical.spp_latest_load_forecast where dt='2018-03-21' ''')

In [6]:
print('forecast wind is',df_wind.MTWF.mean(), 'actual_wind is', actual_wind.Gen_MW.mean())
print('forecast load is',df_load.MTLF.mean(), 'actual_load is', actual_load.actual_load_MW.mean())

forecast wind is 3489.6808333333333 actual_wind is 3698.6949999999997
forecast load is 27506.583333333332 actual_load is 27647.083333333332


In [7]:
# What nuclear plants were out on Mar 19, 2019 in SPP and MISO 
# WATERF 26 30 G3 UN

df_nuclear = sql_functions.download_df_from_sql_db(
    "select n.dt, n.unitId, g.unitName, z.opExchange, z.zone, "
    "n.power as output_MW "
    "from common.NRCNuclearOutput n "
    "join common.NuclearGeneratorId g on n.unitId = g.unitId "
    "join common.zonalNuclearPlant z on n.unitId = z.unitId "
    "where n.dt = '2019-03-19' "
    "and z.opExchange in ('SPP', 'MISO')")

df_capacity = sql_functions.download_df_from_sql_db(
    "select d.unitId, sum(b.capacity) as capacity "
    "from common.TEMPNRCNuclearCapacity b "
    "join common.NuclearGeneratorNameMapping c on b.plant = c.plant "
    "join common.NuclearGeneratorId d on c.unitName = d.unitName "
    "where b.Yr = 2019 group by d.unitId")

df_nuclear['unitId']= pd.to_numeric(df_nuclear['unitId'])
df_merged = df_nuclear.merge(df_capacity, on='unitId')
df_merged['outage_MW'] = df_merged['capacity'] - df_merged['output_MW']
# plants that were partially or fully out
print(df_merged[df_merged['outage_MW']>df_merged['outage_MW'].quantile(0.8)].unitName.unique())

['Quad Cities 1' 'Point Beach 1' 'Monticello' 'Waterford 3' 'D.C. Cook 1']


In [8]:
# Find out the spot gas prices as of Feb 2 and Feb 16, 2021 for Panhandle and Henry Hub
df_gas2 = sql_functions.download_df_from_sql_db(
    "select dt, henryHub, panHandleHub "
    "from odessa_VirtualStrategy_MISO.iceVariables "
    "where dt between '2021-02-02' and '2021-02-16'")
print(df_gas2)

            dt  henryHub  panHandleHub
0   2021-02-02     2.810          2.69
1   2021-02-03     3.150          2.90
2   2021-02-04     3.010          2.88
3   2021-02-05     2.920          2.80
4   2021-02-06     3.475          4.95
5   2021-02-07     3.475          4.95
6   2021-02-08     3.475          4.95
7   2021-02-09     3.400          3.82
8   2021-02-10     3.165          3.70
9   2021-02-11     3.500          4.75
10  2021-02-12     5.250         15.50
11  2021-02-13     7.600         65.00
12  2021-02-14     7.600         65.00
13  2021-02-15     7.600         65.00
14  2021-02-16     7.600         65.00


In [9]:
# For node 1164, what is the mapped load zone(s) and reserve zone.
# Step 1: Get load zone for node 1164
df_node = sql_functions.download_df_from_sql_db(
    "select node_num, node_name, node_zone "
    "from spp_core.spp_settlement_location_node_list "
    "where node_num = 1164")
print(df_node)
load_zone = df_node['node_zone'].iloc[0]
df_reserve = sql_functions.download_df_from_sql_db(
    f"select zone, ReserveZone "
    f"from spp_core.zoneToReserveZoneMapping "
    f"where zone = '{load_zone}'")
print(df_reserve)

   node_num                 node_name node_zone
0      1164  WFEC.PEOP.CENTRAHOMAEAST      WFEC
   zone ReserveZone
0  WFEC           4


In [10]:
# Wind forecast vs actual by reserve zone on Mar 09
df_wind = sql_functions.download_df_from_sql_db(
    "select dt, hr, ReserveZone, WindForecastMW, WindActualMW "
    "from spp_physical.spp_wind_solar_reserveZone_latest "
    "where dt = '2018-03-09' "
    "order by ReserveZone, hr")

# Load forecast vs actual on Mar 09
df_load = sql_functions.download_df_from_sql_db(
    "select dt, hr, load_MW as load_forecast_MW, actual_load_MW "
    "from spp_physical.spp_latest_load_forecast "
    "where dt = '2018-03-09' "
    "order by hr")

# VER curtailments on Mar 09
df_ver = sql_functions.download_df_from_sql_db(
    "select dt, hr, "
    "WindRedispatchCurtailments, WindManualCurtailments, WindCurtailedForEnergy, "
    "WindRedispatchCurtailments + WindManualCurtailments + WindCurtailedForEnergy as TotalWindCurtailments "
    "from spp_physical.ver_curtailments_hourly "
    "where dt = '2018-03-09' "
    "order by hr")

# aggregate wind across all reserve zones per hour
wind_total = df_wind.groupby('hr')[['WindForecastMW', 'WindActualMW']].sum().reset_index()
wind_total['wind_error_MW'] = wind_total['WindActualMW'] - wind_total['WindForecastMW']

# merge all three together
df_load['load_error_MW'] = df_load['actual_load_MW'] - df_load['load_forecast_MW']

summary = wind_total.merge(df_load[['hr', 'load_forecast_MW', 'actual_load_MW', 'load_error_MW']], on='hr') \
                    .merge(df_ver[['hr', 'WindRedispatchCurtailments', 'WindManualCurtailments',
                                   'WindCurtailedForEnergy', 'TotalWindCurtailments']], on='hr')

print(summary.to_string(index=False))


 hr  WindForecastMW  WindActualMW  wind_error_MW  load_forecast_MW  actual_load_MW  load_error_MW  WindRedispatchCurtailments  WindManualCurtailments WindCurtailedForEnergy TotalWindCurtailments
  1        10787.53      11410.42         622.89             27043           27211            168                    95.37580                0.000000                   None                  None
  2        11341.21      11858.68         517.47             26436           26582            146                   167.35200                0.014167                   None                  None
  3        11335.87      11365.94          30.07             26118           26324            206                    45.93920                5.371670                   None                  None
  4        10814.31      10956.26         141.95             26258           26275             17                    20.06670               22.787500                   None                  None
  5        10275.90      

In [11]:
# In the last 2 months, what date and hour was wind generation highest as a percent of total generation. What date and hour was it the lowest. What are RT and DA slack prices in the top 5 hours and bottom 5 hours for wind as a percentage of total generation

from datetime import date, timedelta
two_months_ago = (date.today() - timedelta(days=60)).strftime('%Y-%m-%d')

# ── 1. Wind % of total generation ──
df_gen = sql_functions.download_df_from_sql_db(
    f"select dt, hr, WindTotal, "
    f"CoalTotal + NaturalGasTotal + NuclearTotal + HydroTotal + WindTotal + SolarTotal as TotalGen_MW "
    f"from spp_physical.GenMixHourly "
    f"where dt >= '{two_months_ago}' "
    f"order by dt, hr")

df_gen['wind_pct'] = df_gen['WindTotal'] / df_gen['TotalGen_MW'] * 100

# top 5 and bottom 5 hours by wind %
top5    = df_gen.nlargest(5,  'wind_pct')[['dt', 'hr', 'WindTotal', 'TotalGen_MW', 'wind_pct']]
bottom5 = df_gen.nsmallest(5, 'wind_pct')[['dt', 'hr', 'WindTotal', 'TotalGen_MW', 'wind_pct']]

print("=== Top 5 Wind % Hours ===")
print(top5)
print("\n=== Bottom 5 Wind % Hours ===")
print(bottom5)

# RT and DA slack for those hours use hub node 635 (SPPSOUTH_HUB) for system-wide slack
top5_dates    = tuple(top5['dt'].astype(str).tolist())
bottom5_dates = tuple(bottom5['dt'].astype(str).tolist())
all_dates     = top5_dates + bottom5_dates

df_rt = sql_functions.download_df_from_sql_db(
    f"select dt, hr-1 as hr, "
    f"rt_value - congestional_value - marginalloss_value as rt_slack "
    f"from spp_lmp.settlement_location_rt_hourly "
    f"where node_num = 635 and dt in {all_dates}")

df_da = sql_functions.download_df_from_sql_db(
    f"select dt, hr-1 as hr, "
    f"da_value - congestional_value - marginalloss_value as da_slack "
    f"from spp_lmp.settlement_location_da_hourly "
    f"where node_num = 635 and dt in {all_dates}")


# merge slack into top/bottom
def add_slack(df):
    return df.merge(df_rt, on=['dt','hr']).merge(df_da, on=['dt','hr'])

print("\n=== Top 5 Wind % with RT/DA Slack ===")
print(add_slack(top5.copy()))

print("\n=== Bottom 5 Wind % with RT/DA Slack ===")
print(add_slack(bottom5.copy()))


=== Top 5 Wind % Hours ===
              dt  hr  WindTotal  TotalGen_MW   wind_pct
555   2026-02-17   4    23162.8      31331.1  73.929099
554   2026-02-17   3    22070.4      29965.9  73.651717
551   2026-02-17   0    22283.7      30337.2  73.453384
553   2026-02-17   2    22057.1      30042.0  73.420877
1170  2026-03-14  21    23326.3      31842.9  73.254320

=== Bottom 5 Wind % Hours ===
             dt  hr  WindTotal  TotalGen_MW  wind_pct
926  2026-03-04  15     1570.7      31300.3  5.018163
925  2026-03-04  14     1561.3      30940.0  5.046218
352  2026-02-08  17     1638.1      31273.9  5.237914
351  2026-02-08  16     1745.1      31192.8  5.594560
924  2026-03-04  13     1881.5      31437.2  5.984948

=== Top 5 Wind % with RT/DA Slack ===
           dt  hr  WindTotal  TotalGen_MW   wind_pct  rt_slack  da_slack
0  2026-02-17   4    23162.8      31331.1  73.929099    1.0746   -5.8907
1  2026-02-17   3    22070.4      29965.9  73.651717  -22.1399   -6.8112
2  2026-02-17   0    222

In [ ]:
# Is node 3 valid for Mar 5, 2019
df_node = sql_functions.download_df_from_sql_db(
    "select node_num, node_name, end_date "
    "from spp_core.spp_settlement_location_node_list "
    "where node_num = 3 "
    "and end_date = '2019-03-05'")
print("Node 3 valid:", len(df_node) > 0)
print(df_node)

# Q2: CY2019 total profits and operating reserve charges ──
df_pnl = sql_functions.download_df_from_sql_db(
    "select "
    "  sum(r.profitGross)                                     as total_gross_profit, "
    "  sum(r.clear_mw * o.daMwpDistRate)                     as da_op_reserve_charges, "
    "  sum(r.clear_mw * o.rtMwpDistRate)                     as rt_op_reserve_charges, "
    "  sum(r.clear_mw * (o.daMwpDistRate + o.rtMwpDistRate)) as total_op_reserve_charges, "
    "  sum(r.profitGross) - "
    "  sum(r.clear_mw * (o.daMwpDistRate + o.rtMwpDistRate)) as net_profit "
    "from odessa_Bid.SPPFinalResultsSummarized r "
    "left join spp_virtual.v3_op_reserve_and_fees o "
    "  on r.dt = o.dt and o.source = 'ftp' "
    "where r.dt >= '2019-01-01' and r.dt <= '2019-12-31'")
print(df_pnl)

# Q3: Credit needed for 5MW on nodes 1164 & 1038, Feb 20 2019 ──
df_credit = sql_functions.download_df_from_sql_db(
    "select nodeNum, settlementLocation, bidRefPrice, offerRefPrice, "
    "  5 * bidRefPrice   as credit_needed_bid_5MW, "
    "  5 * offerRefPrice as credit_needed_offer_5MW "
    "from spp_virtual.v3_creditReferencePrices "
    "where nodeNum in (1164, 1038) "
    "and startDt <= '2019-02-20' and endDt >= '2019-02-20'")
print(df_credit)
print("Total credit needed:", df_credit['credit_needed_bid_5MW'].sum())


Node 3 valid: False
Empty DataFrame
Columns: [node_num, node_name, end_date]
Index: []
   total_gross_profit  da_op_reserve_charges  rt_op_reserve_charges  \
0        1.482886e+06            106873.1129            1146223.659   

   total_op_reserve_charges   net_profit  
0              1.253097e+06  229789.2903  
   nodeNum        settlementLocation  bidRefPrice  offerRefPrice  \
0     1164  WFEC.PEOP.CENTRAHOMAEAST       -27.14         -47.92   
1     1038            WFEC_PEOP_LOAD       -25.44         -60.31   

   credit_needed_bid_5MW  credit_needed_offer_5MW  
0            -135.699997              -239.599991  
1            -127.200003              -301.550007  
Total credit needed: -262.8999996185303


: 

: 

: 

In [16]:
query_top25 = """
SELECT 
    c.ConstraintNum,
    n3.monitoredName   AS monitored_name,
    n4.contingencyName AS contingency_name,
    SUM(c.MValue)      AS total_mvalue
FROM spp_constraints.{table} c
JOIN spp_constraints.ConstraintMaster_v3     n1 ON c.ConstraintNum = n1.constraintNum
JOIN spp_constraints.MonitoredMaster_v3      n3 ON n1.monitoredNum = n3.monitoredNum
JOIN spp_constraints.ContingencyMaster_v3    n4 ON n1.contingencyNum = n4.contingencyNum
WHERE c.dt >= DATE_SUB(CURDATE(), INTERVAL 1 YEAR)
GROUP BY 1,2,3
ORDER BY total_mvalue DESC
LIMIT 25
"""

df_top25_da = sql_functions.download_df_from_sql_db(query_top25.format(table='congestion_da_hourly_v3'))
df_top25_rt = sql_functions.download_df_from_sql_db(query_top25.format(table='congestion_rt_hourly_v3'))
print("=== TOP 25 DA ===");  print(df_top25_da)
print("=== TOP 25 RT ===");  print(df_top25_rt)


=== TOP 25 DA ===
    ConstraintNum                  monitored_name  \
0          582318        LN MIDLTNTP - CRSW_REV23   
1          414970             LN GAVINS - SPIRITM   
2          554090            LN E_11_ST - CATOOSA   
3          819866      LN DAKOTA - CNTRVILL_REV23   
4          740626             LN BRANNW5 - BRN331   
5          818214            LN ANTA_TP - EXIRAWA   
6          776597            LN ALTO - TIOG_REV23   
7          774539      LN LEWISWP - DAWSONC_REV23   
8          835281       LN NRTH_INT - WAITS_REV23   
9          521447           LN ATL1091 - CARTHAGE   
10         554328            LN WINSLOW - OAKLAND   
11         567370      LN CLARKS4 - OZARKH5_REV23   
12         843302          XFMR SHAM - SHAM_REV23   
13         842090  XFMR POTTER_S - POTTER_S_REV23   
14         864421           LN MILLC - ARB1_REV23   
15         788011           LN HOCKERVL - MIAMI_1   
16         778579           LN WHEELER - SWEETWT6   
17         857508          L